In [ ]:
# ==========================================
# 2. PROCESAMIENTO EN LOTE (S2 + S1 + EMAIL + WHATSAPP)
# ==========================================
import smtplib # Aseguramos que la librería para enviar correos esté disponible

datos = s_reg.get_all_values()
filas_procesadas = 0

for i, fila in enumerate(datos):
    if i == 0: continue # Saltea los encabezados de la fila 1

    # Aseguramos espacio en la lista para leer y escribir hasta la columna S (índice 18) sin desbordes
    while len(fila) < 19: fila.append('')

    # --- LECTURA DE VARIABLES INICIALES ---
    fecha_a = str(fila[0]).strip()                 # Columna A: FECHA (Índice 0)
    partida_cruda = str(fila[11]).strip()          # Columna L: PARTIDA (Índice 11)
    nombre_cliente = str(fila[12]).strip()         # Columna M: NOMBRE (Índice 12)
    mail_cliente = str(fila[13]).strip()           # Columna N: EMAIL (Índice 13)
    lat_cruda = str(fila[15]).strip().replace(',', '.') # Columna P: LATITUD (Índice 15)
    lon_cruda = str(fila[16]).strip().replace(',', '.') # Columna Q: LONGITUD (Índice 16)
    celular_cliente = str(fila[18]).strip()        # Columna S: CELULAR WHATSAPP (Índice 18)

    # Verificamos si la fila es válida para procesar
    if not fecha_a and ((partida_cruda and partida_cruda != 'None' and partida_cruda != '') or (lat_cruda and lon_cruda)):
        target_row = i + 1

        catastro = ee.FeatureCollection(RUTA_CATASTRO)
        lote_filtrado = None

        # --- BÚSQUEDA CATASTRAL ---
        if partida_cruda and partida_cruda != 'None' and partida_cruda != '':
            partida_limpia = partida_cruda.split('.')[0].strip()
            partida_lote = partida_limpia.zfill(6)
            print(f"\n🚜 Procesando Fila {target_row} por PARTIDA: {partida_lote} | Cliente: {nombre_cliente}")
            lote_filtrado = catastro.filter(ee.Filter.stringContains('PDA', partida_limpia))
        else:
            try:
                lat = float(lat_cruda)
                lon = float(lon_cruda)
                print(f"\n🎯 Procesando Fila {target_row} por COORDENADAS: Lat {lat} | Lon {lon} | Cliente: {nombre_cliente}")
                punto_busqueda = ee.Geometry.Point([lon, lat])
                lote_filtrado = catastro.filterBounds(punto_busqueda)
                partida_lote = f"Ubicación [{lat}, {lon}]"
            except ValueError:
                print(f"❌ ERROR: Las coordenadas en la fila {target_row} no son números válidos.")
                continue

        if lote_filtrado is None or lote_filtrado.size().getInfo() == 0:
            print(f"❌ ATENCIÓN: El lote de la fila {target_row} no se encontró en el mapa. Verificalo.")
            continue

        lote_real = lote_filtrado.first()
        geometria_lote = lote_real.geometry()

        if not partida_cruda or partida_cruda == 'None' or partida_cruda == '':
            try:
                pda_interna = lote_real.get('PDA').getInfo()
                if pda_interna:
                    partida_lote = str(pda_interna).split('.')[0].strip().zfill(6)
            except:
                pass

        # --- INTENTAR EXTRACCIÓN OPTICA (SENTINEL-2) ---
        hoy = datetime.now()
        dias_atras = hoy - timedelta(days=5)
        modo_satelite = "Sentinel-2"

        coleccion_s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                .filterBounds(geometria_lote) \
                .filterDate(dias_atras.strftime('%Y-%m-%d'), (hoy + timedelta(days=1)).strftime('%Y-%m-%d')) \
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
                .sort('system:time_start', False)

        imagen_mas_nueva = coleccion_s2.first()

        # --- VERIFICACIÓN DE NUBES Y PROTOCOLO FALLBACK DE RADAR (SENTINEL-1) ---
        if imagen_mas_nueva.getInfo() is None:
            print("☁️ ALERTA: Lote 100% nublado para Sentinel-2. Activando Protocolo de Radar Sentinel-1...")
            modo_satelite = "Sentinel-1 (Radar)"

            coleccion_s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
                .filterBounds(geometria_lote) \
                .filterDate(dias_atras.strftime('%Y-%m-%d'), (hoy + timedelta(days=1)).strftime('%Y-%m-%d')) \
                .filter(ee.Filter.eq('instrumentMode', 'IW')) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
                .sort('system:time_start', False)

            imagen_mas_nueva = coleccion_s1.first()
            if imagen_mas_nueva.getInfo() is None:
                print("❌ ERROR CRÍTICO: Tampoco hay datos de radar disponibles. Saltando fila.")
                continue

        # Extraer fecha de captura
        fecha_foto_ms = imagen_mas_nueva.get('system:time_start').getInfo()
        fecha_foto = datetime.fromtimestamp(fecha_foto_ms / 1000.0).strftime('%d/%m/%Y')

        # Variables listas por defecto para las celdas
        ndvi_val, evi_val, ndwi_val, savi_val, gndvi_val, ndre_val = "0", "0", "0", "0", "0", "0"
        rvi_val, ratio_val, vv_val = "0", "0", "0"

        # --- RAMAL DE PROCESAMIENTO TÉCNICO SEGÚN EL SATÉLITE QUE GANÓ ---
        if modo_satelite == "Sentinel-2":
            imagen_ndvi = imagen_mas_nueva.normalizedDifference(['B8', 'B4']).rename('NDVI')
            vis_params_ndvi = {'min': 0.1, 'max': 0.8, 'palette': ['red', 'yellow', 'green']}
            url_imagen_satelite = imagen_ndvi.visualize(**vis_params_ndvi).getThumbURL({
                'region': geometria_lote, 'dimensions': 600, 'format': 'jpg'
            })

            res = imagen_mas_nueva.reduceRegion(
                reducer=ee.Reducer.mean(), geometry=geometria_lote, scale=10, bestEffort=True
            ).getInfo()

            nir = res.get('B8', 0) / 10000
            red = res.get('B4', 0) / 10000
            green = res.get('B3', 0) / 10000
            blue = res.get('B2', 0) / 10000
            red_edge = res.get('B5', 0) / 10000

            ndvi = (nir - red) / (nir + red + 0.0001)
            ndwi = (green - nir) / (green + nir + 0.0001)
            divisor_evi = nir + 6 * red - 7.5 * blue + 1
            evi = 2.5 * ((nir - red) / divisor_evi) if divisor_evi != 0 else 0
            divisor_savi = nir + red + 0.5
            savi = ((nir - red) / divisor_savi) * 1.5 if divisor_savi != 0 else 0
            gndvi = (nir - green) / (nir + green + 0.0001)
            ndre = (nir - red_edge) / (nir + red_edge + 0.0001)

            ndvi_val = str(round(ndvi, 3)).replace('.', ',')
            evi_val = str(round(evi, 3)).replace('.', ',')
            ndwi_val = str(round(ndwi, 3)).replace('.', ',')
            savi_val = str(round(savi, 3)).replace('.', ',')
            gndvi_val = str(round(gndvi, 3)).replace('.', ',')
            ndre_val = str(round(ndre, 3)).replace('.', ',')

            est = "Verde" if ndvi > 0.6 else "Amarillo"

            instrucciones_ia = f"""
            Actúa como un Ingeniero Agrónomo Senior. Analiza los índices ópticos medidos por Sentinel-2 del lote (Identificador: {partida_lote} | Foto del {fecha_foto}):
            - NDVI: {round(ndvi, 3)} | GNDVI: {round(gndvi, 3)} | NDRE: {round(ndre, 3)} | EVI: {round(evi, 3)} | SAVI: {round(savi, 3)} | NDWI: {round(ndwi, 3)}
            Desarrolla obligatoriamente un párrafo completo para cada bloque:
            1. ESTADO GENERAL: Significado de NDVI, EVI y GNDVI para vigor y densidad fotosintética.
            2. ANÁLISIS TÉCNICO: Relación entre estrés hídrico (NDWI), nitrógeno (NDRE) y ajuste de suelo (SAVI).
            3. RECOMENDACIÓN DE MANEJO: Pasos agronómicos claros a seguir en el campo.
            REGLA ESTRICTA: Sin saludos, firmas ni despedidas. Termina directo en la recomendación.
            """

        else:
            # Procesamiento de Radar Avanzado (Sentinel-1 Fallback)
            vv_db_img = imagen_mas_nueva.select('VV')
            vh_db_img = imagen_mas_nueva.select('VH')

            vv_lin = ee.Image(10.0).pow(vv_db_img.divide(10.0)).rename('VV')
            vh_lin = ee.Image(10.0).pow(vh_db_img.divide(10.0)).rename('VH')
            rvi_img = vh_lin.multiply(4).divide(vv_lin.add(vh_lin)).rename('RVI')

            img_linear = vv_lin.addBands(vh_lin).addBands(rvi_img)

            res_s1 = img_linear.reduceRegion(
                reducer=ee.Reducer.mean(), geometry=geometria_lote, scale=10, bestEffort=True
            ).getInfo()

            vv_mean_lin = res_s1.get('VV', 0.0001)
            vh_mean_lin = res_s1.get('VH', 0.0001)
            rvi_mean = res_s1.get('RVI', 0)

            vv_db = 10 * math.log10(vv_mean_lin) if vv_mean_lin > 0 else -15
            vh_db = 10 * math.log10(vh_mean_lin) if vh_mean_lin > 0 else -20
            ratio_vh_vv = vh_mean_lin / vv_mean_lin

            rvi_val = str(round(rvi_mean, 3)).replace('.', ',')
            ratio_val = str(round(ratio_vh_vv, 3)).replace('.', ',')
            vv_val = str(round(vv_db, 2)).replace('.', ',')
            est = "Radar (Nublado)"

            url_imagen_satelite = imagen_mas_nueva.visualize(bands=['VV', 'VH', 'VV'], min=-25, max=0).getThumbURL({
                'region': geometria_lote, 'dimensions': 600, 'format': 'jpg'
            })

            instrucciones_ia = f"""
            Actúa como un Ingeniero Agrónomo Senior experto en Teledetección de Radar. El cielo presentaba nubosidad total, por lo que analizamos el lote con Microondas Activas de Radar Sentinel-1 (Identificador: {partida_lote} | Datos del {fecha_foto}):
            - RVI (Radar Vegetation Index): {round(rvi_mean, 3)} | Ratio VH/VV (Biomasa Estructural): {round(ratio_vh_vv, 3)} | Retrodispersión VV (Humedad de Suelo/Planta): {round(vv_db, 2)} dB | Retrodispersión VH (Densidad de Tallos): {round(vh_db, 2)} dB
            Desarrolla obligatoriamente un párrafo completo para cada bloque:
            1. ESTADO GENERAL: Qué significa este nivel de RVI y el Ratio VH/VV para la cantidad de biomasa, densidad de tallos y estructura física desarrollada por el cultivo.
            2. ANÁLISIS TÉCNICO: Analiza el comportamiento hídrico del suelo y canopeo según el coeficiente VV y evalúa si la densidad de rugosidad VH muestra anomalías o retrasos de desarrollo.
            3. RECOMENDACIÓN DE MANEJO: Qué acciones de manejo físico, fertilización o monitoreo terrestre debe ejecutar el productor considerando que el cielo está cubierto pero el radar detectó estos niveles estructurales.
            REGLA ESTRICTA: Sin saludos, firmas ni despedidas. Termina directo en la recomendación.
            """

        print(f"🔗 Link temporal de la foto generada ({modo_satelite}): {url_imagen_satelite}")

        # --- CONSULTA INTELIGENCIA ARTIFICIAL ---
        print("🤖 Consultando a Gemini 2.5 Flash...")
        try:
            respuesta_ia = client.models.generate_content(model='gemini-2.5-flash', contents=instrucciones_ia)
            diagnostico_final = respuesta_ia.text.strip()
        except Exception as e:
            diagnostico_final = "Error al generar diagnóstico con IA."
            print(f"⚠️ ATENCIÓN - EL ERROR REAL DE GEMINI EN LA FILA {target_row} FUE: {e}")

        # --- GUARDAR EN EXCEL (TABLERO PREMIUM DE 11 COLUMNAS DE CONTROL) ---
        s_reg.update_cell(target_row, 1, hoy.strftime("%d/%m/%Y")) # A: Fecha
        s_reg.update_cell(target_row, 2, ndvi_val)                 # B: NDVI
        s_reg.update_cell(target_row, 3, evi_val)                  # C: EVI
        s_reg.update_cell(target_row, 4, ndwi_val)                 # D: NDWI
        s_reg.update_cell(target_row, 5, savi_val)                 # E: SAVI
        s_reg.update_cell(target_row, 6, gndvi_val)                # F: GNDVI
        s_reg.update_cell(target_row, 7, ndre_val)                 # G: NDRE
        s_reg.update_cell(target_row, 8, rvi_val)                  # H: RVI (Radar)
        s_reg.update_cell(target_row, 9, ratio_val)                # I: VH/VV (Radar)
        s_reg.update_cell(target_row, 10, vv_val)                  # J: Humedad VV (Radar)
        s_reg.update_cell(target_row, 11, est)                     # K: Estado
        s_reg.update_cell(target_row, 18, diagnostico_final)       # R: Diagnóstico IA

        # --- ENVIAR CORREO DESDE TU CUENTA DE AGENCIA ---
        if mail_cliente and "@" in mail_cliente:
            try:
                remitente = "update.studiob.juarez@gmail.com"
                password_aplicacion = "wugpzidmctyycnkb"

                msg = MIMEMultipart('related')
                msg['Subject'] = Header(f"📋 Reporte {modo_satelite} de Monitoreo Catastral - Partida {partida_lote}", 'utf-8')
                msg['From'] = remitente
                msg['To'] = mail_cliente
                msg['Bcc'] = "update.studiob.juarez@gmail.com"

                diagnostico_html = diagnostico_final.replace('\n', '<br>')

                if "Radar" in modo_satelite:
                    texto_explicativo = f"Debido a condiciones de nubosidad continua sobre Benito Juárez, hemos activado nuestro sistema avanzado de <strong>Radar de Microondas Sentinel-1</strong>. Este reporte analiza la estructura física, volumen de biomasa y humedad del lote atravesando las nubes por completo."
                    pie_foto = f"Mapa de rugosidad y estructura de biomasa por Radar activo (Sentinel-1) procesado el {fecha_foto}. El radar no se ve afectado por factores climáticos y garantiza el monitoreo en días nublados."
                else:
                    texto_explicativo = f"Te adjuntamos el reporte agrometeorol&oacute;gico procesado el d&iacute;a de la fecha sobre los l&iacute;mites geogr&aacute;ficos oficiales de tu lote (Partida: <strong>{partida_lote}</strong>)."
                    pie_foto = f"Última captura satelital libre de nubosidad (Sentinel-2) procesada el {fecha_foto}. Recorte exacto sobre Nomenclatura Catastral (ARBA) con filtro automático de nubes para garantizar precisión en los índices."

                cuerpo_mail_html = f"""
                <html>
                <body style="font-family: Arial, sans-serif; color: #333; line-height: 1.6; max-width: 600px; margin: auto;">
                    <h2 style="color: #2E7D32;">Hola {nombre_cliente},</h2>
                    <p>{texto_explicativo}</p>

                    <div style="text-align: center; margin: 20px 0;">
                        <img src="{url_imagen_satelite}" alt="Imagen Satelital del Lote" style="width: 100%; max-width: 500px; border: 3px solid #4CAF50; border-radius: 8px; box-shadow: 0px 4px 8px rgba(0,0,0,0.2);">
                        <p style="font-size: 12px; color: #666;"><strong>{pie_foto}</strong></p>

                        <div style="background-color: #f9f9f9; border: 1px solid #ddd; border-radius: 6px; padding: 10px; margin-top: 15px; display: inline-block; text-align: left;">
                            <p style="margin: 0 0 5px 0; font-size: 12px; font-weight: bold; color: #333;">Referencias de Lectura:</p>
                            <p style="margin: 2px 0; font-size: 12px; color: #555;">🛰️ <strong>Modo de Medición:</strong> Ejecutado vía {modo_satelite}.</p>
                            <p style="margin: 2px 0; font-size: 12px; color: #555;">📊 El diagnóstico agronómico inferior desmenuza los valores exactos sobre tus límites catastrales.</p>
                        </div>
                    </div>

                    <div style="background-color: #F1F8E9; padding: 15px; border-left: 4px solid #4CAF50; border-radius: 4px;">
                        {diagnostico_html}
                    </div>

                    <hr style="border: 0; border-top: 1px solid #ccc; margin-top: 30px;">
                    <div style="text-align: center; color: #888; font-family: Arial, sans-serif;">
                        <img src="https://i.postimg.cc/QCL4hXLZ/Gemini-Generated-Image-6awbzt6awbzt6awb.png" alt="Update Studio Logo" style="width: 73px; height: auto; margin-bottom: 6px;">
                        <p style="margin: 0; font-size: 15px; font-weight: bold; color: #2E7D32;">Update Studio</p>
                        <p style="margin: 5px 0 0 0; font-size: 12px;">Monitoreo Avanzado de Precisi&oacute;n Satelital - Sentinel-1 & Sentinel-2 - IA</p>
                    </div>
                </body>
                </html>
                """

                msg.attach(MIMEText(cuerpo_mail_html, 'html', 'utf-8'))

                # --- EL CÓDIGO SMTP PARA ENVIAR REALMENTE EL EMAIL ---
                servidor_correo = smtplib.SMTP('smtp.gmail.com', 587)
                servidor_correo.starttls()
                servidor_correo.login(remitente, password_aplicacion)
                servidor_correo.send_message(msg)
                servidor_correo.quit()
                print(f"📧 Correo despachado con éxito a {mail_cliente}")

            except Exception as e:
                print(f"❌ Error al enviar correo a {mail_cliente}: {e}")

        # =====================================================================
        # 4. BLOQUE DE WHATSAPP AUTOMÁTICO ADAPTADO Y ACOTADO
        # =====================================================================
        # Asegúrate de que ACTIVAR_WHATSAPP esté definida en tu configuración global (True/False)
        if ACTIVAR_WHATSAPP and celular_cliente and celular_cliente != 'None' and celular_cliente != '':
            try:
                print(f"📱 Despachando WhatsApp automático acotado a {celular_cliente}...")

                # 1. Armamos el encabezado profesional e híbrido
                texto_encabezado = f"🌱 *Update Studio - Reporte Satelital*\n"
                texto_encabezado += f"📋 *Identificador:* {partida_lote}\n"
                texto_encabezado += f"📅 *Captura:* {fecha_foto}\n"
                texto_encabezado += f"🛰️ *Modo:* {modo_satelite}\n\n"

                texto_encabezado += f"📊 *Índices de Control:*\n"

                if modo_satelite == "Sentinel-2":
                    texto_encabezado += f"• NDVI (Salud): {ndvi_val}\n"
                    texto_encabezado += f"• NDWI (Humedad): {ndwi_val}\n"
                    texto_encabezado += f"• EVI (Biomasa): {evi_val}\n\n"
                else:
                    texto_encabezado += f"• RVI (Vigor Radar): {rvi_val}\n"
                    texto_encabezado += f"• VH/VV (Estructura): {ratio_val}\n"
                    texto_encabezado += f"• Humedad Suelo: {vv_val} dB\n\n"

                texto_encabezado += f"🤖 *Resumen del Diagnóstico (IA):*\n\n"

                # 2. Cortamos el texto de la IA
                if len(diagnostico_final) > 350:
                    diagnostico_corto = diagnostico_final[:350] + "...\n\n📧 *Nota: El informe detallado fue enviado a tu correo electrónico.*"
                else:
                    diagnostico_corto = diagnostico_final

                # 3. Unimos las partes
                texto_final_whatsapp = texto_encabezado + diagnostico_corto

                # 4. Envío de datos a UltraMsg
                url_whatsapp = "https://api.ultramsg.com/instance179510/messages/image"
                payload = {
                    "token": "7u0jzcpbgm527i2m",
                    "to": celular_cliente,
                    "image": url_imagen_satelite,
                    "caption": texto_final_whatsapp
                }
                headers = {'content-type': 'application/x-www-form-urlencoded'}

                respuesta = requests.post(url_whatsapp, data=payload, headers=headers)
                print(f"✅ ¡WhatsApp enviado con éxito a {celular_cliente}!")

            except Exception as wa_error:
                print(f"❌ Error en el módulo de WhatsApp: {wa_error}")

        filas_procesadas += 1
        time.sleep(12)

if filas_procesadas == 0:
    print("\n🤷‍♂️ No hay parcelas nuevas para procesar o las ingresadas no existen en el catastro.")
else:
    print(f"\n🎉 ¡Proceso terminado! Se analizaron y despacharon {filas_procesadas} reportes automáticos premium con tablero separado.")


🚜 Procesando Fila 2 por PARTIDA: 014609 | Cliente: Guillermo
☁️ ALERTA: Lote 100% nublado para Sentinel-2. Activando Protocolo de Radar Sentinel-1...
🔗 Link temporal de la foto generada (Sentinel-1 (Radar)): https://earthengine.googleapis.com/v1/projects/update-studio-premium/thumbnails/aaaec845e4452be20fee8dfa7beb07db-f5a5ed79cde3060eb30863efcea998d7:getPixels
🤖 Consultando a Gemini 2.5 Flash...
⚠️ ATENCIÓN - EL ERROR REAL DE GEMINI EN LA FILA 2 FUE: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}
📧 Correo despachado con éxito a guille8008@gmail.com
📱 Despachando WhatsApp automático acotado a 5492281310771...
✅ ¡WhatsApp enviado con éxito a 5492281310771!

🎯 Procesando Fila 3 por COORDENADAS: Lat -37.63947 | Lon -59.81325 | Cliente: Agraria
☁️ A